In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# --- (Assuming 'all_battery_data_EIS' and 'col_names' exist) ---
# Make sure your 'circuit_model' (e.g., L1-R0-p(R1,CPE1)-W1)
# parameters are correctly listed here.
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']
# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING (Prerequisite)
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points from 0.01 Hz to 10 kHz

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)')
        
        f_x = raw_df['Frequency(Hz)'].values
        R = raw_df['R(ohm)'].values
        X = raw_df['X(ohm)'].values
        
        # Create interpolation functions
        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        
        # Get features at our fixed frequencies
        R_features = interp_R(fixed_freqs)
        X_features = interp_X(fixed_freqs)
        
        # Combine R and X to make one long feature vector
        features = np.concatenate([R_features, X_features])
        eis_features_list.append(features)
        
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None) # Add a placeholder

# 4. Create the final aligned X and y DataFrames
r_cols = [f'R_{freq:.2f}Hz' for freq in fixed_freqs]
x_cols = [f'X_{freq:.2f}Hz' for freq in fixed_freqs]

X_features_df = pd.DataFrame(eis_features_list, columns=r_cols + x_cols)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']] # Use [['SoH']] to keep it as a DataFrame

# 5. Drop any rows that failed feature engineering
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

print(f"Data ready: X_features ({X_features_final.shape}), y_params ({y_params_final.shape}), y_soh ({y_soh_final.shape})")

# ===================================================================
# PHASE 1: PYTORCH SETUP (Dataset & Model Architecture)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 1. Scale features
# Scaling is CRITICAL for neural networks
scaler_X = StandardScaler()
scaler_y_params = StandardScaler()

X_scaled = scaler_X.fit_transform(X_features_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final)
y_soh_values = y_soh_final.values # No scaling needed for SoH (already 0-1)

# 2. Train/Test Split
(X_train, X_val, 
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_scaled, y_params_scaled, y_soh_values, test_size=0.2, random_state=42
)

# 3. Custom PyTorch Dataset
class EISDataset(Dataset):
    def __init__(self, features, params, soh):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. Define the Multi-Head Model
class MultiHeadEISModel(nn.Module):
    def __init__(self, input_size, num_params):
        super(MultiHeadEISModel, self).__init__()
        
        # Shared Body
        self.shared_body = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Head 1: Parameter Prediction
        self.param_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_params) # No activation (linear output for regression)
        )
        
        # Head 2: SoH Prediction
        self.soh_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # No activation (linear output for regression)
            nn.Sigmoid() # Add Sigmoid to bound output between 0 and 1
        )

    def forward(self, x):
        # Pass input through the shared body
        shared_output = self.shared_body(x)
        
        # Pass shared output to each head
        params_pred = self.param_head(shared_output)
        soh_pred = self.soh_head(shared_output)
        
        return params_pred, soh_pred

# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

# 1. Setup Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

INPUT_SIZE = X_features_final.shape[1]
NUM_PARAMS = y_params_final.shape[1]
NUM_EPOCHS = 50 # Increase this for better results
LEARNING_RATE = 0.001

model = MultiHeadEISModel(INPUT_SIZE, NUM_PARAMS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Define your loss weights
alpha = 0.7 # Weight for SoH loss (as per your example)
beta = 0.3  # Weight for Parameter loss

# 3. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for features, true_params, true_soh in train_loader:
        features = features.to(device)
        true_params = true_params.to(device)
        true_soh = true_soh.to(device)
        
        # --- Forward Pass ---
        pred_params, pred_soh = model(features)
        
        # --- Calculate Combined Loss ---
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh) # .view(-1, 1) not needed, shape is correct
        
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        # --- Backward Pass ---
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        
        total_train_loss += loss_total.item()

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    total_val_soh_loss = 0
    total_val_params_loss = 0
    
    with torch.no_grad():
        for features, true_params, true_soh in val_loader:
            features = features.to(device)
            true_params = true_params.to(device)
            true_soh = true_soh.to(device)
            
            pred_params, pred_soh = model(features)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh)
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# --- 4. Final Evaluation on Test Set ---
model.eval()
with torch.no_grad():
    # Predict on the entire validation set
    all_features = torch.tensor(X_val, dtype=torch.float32).to(device)
    true_params = torch.tensor(y_params_val, dtype=torch.float32).to(device)
    true_soh = torch.tensor(y_soh_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_features)
    
    # --- De-scale the predictions to be human-readable ---
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    
    # --- Calculate Final Errors ---
    soh_rmse = np.sqrt(mean_squared_error(true_soh.cpu().numpy(), pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    
    # Calculate RMSE for each parameter
    for i, name in enumerate(param_columns):
        # Compare the i-th column of y_params_val with the i-th column of pred_params
        param_rmse = np.sqrt(mean_squared_error(y_params_val[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")

--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 1815.43it/s]


Data ready: X_features ((549, 100)), y_params ((549, 6)), y_soh ((549, 1))

--- Phase 1: Setting up PyTorch components ---

--- Phase 2: Starting Model Training ---
Using device: cpu
Epoch [01/50] - Train Loss: 0.331180 | Val Loss: 0.303126 (SoH: 0.017469, Params: 0.969658)
Epoch [02/50] - Train Loss: 0.226469 | Val Loss: 0.278498 (SoH: 0.013371, Params: 0.897128)
Epoch [03/50] - Train Loss: 0.217599 | Val Loss: 0.261407 (SoH: 0.010015, Params: 0.847988)
Epoch [04/50] - Train Loss: 0.209355 | Val Loss: 0.264803 (SoH: 0.006490, Params: 0.867535)
Epoch [05/50] - Train Loss: 0.177053 | Val Loss: 0.252702 (SoH: 0.004871, Params: 0.830974)
Epoch [06/50] - Train Loss: 0.168075 | Val Loss: 0.250212 (SoH: 0.003755, Params: 0.825277)
Epoch [07/50] - Train Loss: 0.164040 | Val Loss: 0.247352 (SoH: 0.002623, Params: 0.818387)
Epoch [08/50] - Train Loss: 0.165270 | Val Loss: 0.246505 (SoH: 0.002002, Params: 0.817011)
Epoch [09/50] - Train Loss: 0.159599 | Val Loss: 0.246168 (SoH: 0.001753, Params: